In [ ]:
%%configure -f
{
  "driverMemory": "28g",
  "executorMemory": "28g",
  "conf": {
    "spark.microsoft.delta.parquet.vorder.enabled": "true",
    "spark.sql.parquet.enableVectorizedReader": "true",
    "spark.native.enabled": "true"
  }
}

# Gold Layer - Aggregated Analytics

This notebook consumes the curated data from the Silver layer and performs domain-specific aggregations for reporting and analytics.

**Actions:**
- Aggregating metrics by Region and Product Category
- Creating a Gold Delta table optimized for read queries

In [ ]:
from pyspark.sql import functions as F

# Identify silver table
silver_table = "silver.curated_invoices"
try:
    silver_df = spark.table(silver_table)
except Exception as e:
    silver_table = "silver_curated_invoices"
    silver_df = spark.table(silver_table)

print(f"Reading from: {silver_table}")

In [ ]:
# Aggregations: Revenue by Region and Product Category
gold_df = silver_df.groupBy("region", "product_category").agg(
    F.count("invoice_id").alias("total_transactions"),
    F.sum("calculated_revenue").alias("total_revenue"),
    F.avg("calculated_revenue").alias("avg_revenue_per_invoice"),
    F.max("calculated_revenue").alias("max_revenue_invoice"),
    F.min("calculated_revenue").alias("min_revenue_invoice")
)

gold_df.show(10, truncate=False)

In [ ]:
# Write to Gold layer
def lakehouse_is_schema_enabled():
    import uuid
    probe = f"_schema_probe_{uuid.uuid4().hex}"
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {probe}")
        spark.sql(f"DROP SCHEMA IF EXISTS {probe}")
        return True
    except Exception as e:
        if "Feature not supported on Apache Spark in Microsoft Fabric" in str(e):
            return False
        raise

schema_enabled = lakehouse_is_schema_enabled()

if schema_enabled:
    spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
    gold_table = "gold.sales_analytics"
else:
    gold_table = "gold_sales_analytics"

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_table)

print(f"✅ Gold table written: {gold_table}")